In [ ]:
pip install pyarrow

^C
Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install fire

Note: you may need to restart the kernel to use updated packages.


In [ ]:
pip install transformers

Note: you may need to restart the kernel to use updated packages.


In [ ]:
conda install git

error: incomplete escape \U at position 28

In [ ]:
%pip install git+https://github.com/EleutherAI/lm-evaluation-harness.git

  Cloning https://github.com/EleutherAI/lm-evaluation-harness.git to /tmp/pip-req-build-vechyw2r
  Running command git clone --filter=blob:none --quiet https://github.com/EleutherAI/lm-evaluation-harness.git /tmp/pip-req-build-vechyw2r
  Resolved https://github.com/EleutherAI/lm-evaluation-harness.git to commit 4890e881031a8ff00fd3136f938c4cf1ae101de4
  Installing build dependencies ... one
  Getting requirements to build wheel ... one
  Preparing metadata (pyproject.toml) ... one
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 54.0 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 37.2 MB/s eta 0:00:00 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 38.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━╸━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
pip install git+https://github.com/lxuechen/private-transformers.git

In [ ]:
# Using PyTorch 1.4

import numpy as np
import pyarrow.parquet as pq
import pandas as pd
import random
import torch
import fire
import logging
import os
import csv
import re
import shutil

from torch.utils.data import Dataset, DataLoader
from transformers import GPT2Tokenizer, GPT2LMHeadModel, AdamW, get_linear_schedule_with_warmup
from torch.optim import SGD
from tqdm import tqdm, trange
import torch.nn.functional as F
from private_transformers import PrivacyEngine
from torch.nn.utils.rnn import pad_sequence


E0000 00:00:1741200608.592949    3681 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1741200608.597516    3681 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


In [ ]:

YAML_boolq_string = """
group: multimedqa
task:
  - pubmedqa
  - medmcqa
  - medqa_4options
  - task: mmlu_anatomy
    task_alias: "anatomy (mmlu)"
  - task: mmlu_clinical_knowledge
    task_alias: "clinical_knowledge (mmlu)"
  - task: mmlu_college_medicine
    task_alias: "college_medicine (mmlu)"
  - task: mmlu_medical_genetics
    task_alias: "medical_genetics (mmlu)"
  - task: mmlu_professional_medicine
    task_alias: "professional_medicine (mmlu)"
  - task: mmlu_college_biology
    task_alias: "college_biology (mmlu)"
aggregate_metric_list:
  - metric: acc
    aggregation: mean
    weight_by_size: True
"""
with open("multimedqa.yaml", "w") as f:
    f.write(YAML_boolq_string)

In [ ]:
loss_fn = torch.nn.CrossEntropyLoss(reduction="none") 

def loss_function(outputs, input_tensor):
    logits = outputs.logits  # (batch_size, seq_len, vocab_size)
    shift_logits = logits[:, :-1, :].contiguous()  # Alinear la secuencia
    shift_labels = input_tensor[:, 1:].contiguous() 

    loss_per_example = F.cross_entropy(
        shift_logits.view(-1, shift_logits.size(-1)), 
        shift_labels.view(-1),  # (batch_size * seq_len)
        reduction="none" 
    )

    return loss_per_example.view(input_tensor.size(0), -1).mean(dim=1)  


In [ ]:
import torch
import csv
import os
from tqdm import tqdm
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from torch.optim import SGD
from transformers import GPT2Tokenizer, get_linear_schedule_with_warmup
device = "cuda"


class CSVMIMIC(Dataset):
    def __init__(self, control_code, truncate=False, gpt2_type="gpt2", max_length=768):
        self.tokenizer = GPT2Tokenizer.from_pretrained(gpt2_type)
        self.tokenizer.pad_token = self.tokenizer.eos_token  # Usar EOS como padding
        self.notes = []


        with open("MIMIC_NoteEvents_deanonymized.csv", newline="", encoding="ISO-8859-1") as csvfile:
            tweet_csv = csv.reader(csvfile)
            for row in tweet_csv:
                encoded = self.tokenizer.encode(f"<|{control_code}|>{row[10][:max_length]}<|endoftext|>")
                self.notes.append(torch.tensor(encoded, dtype=torch.long))  # Convertir a tensor long

        if truncate:
            self.notes = self.notes[:20000]

    def __len__(self):
        return len(self.notes)

    def __getitem__(self, idx):
        return self.notes[idx]

# Función para aplicar padding en DataLoader
def collate_fn(batch):
    batch_padded = pad_sequence(batch, batch_first=True, padding_value=tokenizer.pad_token_id)
    return batch_padded.to(device)


def train(
    dataset,
    model,
    batch_size=4,
    epochs=4,
    lr=3e-1,
    warmup_steps=2500,
    device="cuda",
    output_dir=".",
    output_prefix="wreckgar",
    save_model_on_epoch=False,
):
    model = model.to(device)
    model.train()

    optimizer = SGD(model.parameters(), lr=lr)

    privacy_engine = PrivacyEngine(
        model,
        batch_size=batch_size,
        sample_size=len(dataset),
        epochs=epochs,
        max_grad_norm=0.1,
        target_epsilon=8,
        target_delta = (1/len(dataset))*2,
        clipping_mode="per_layer",
    )
    privacy_engine.attach(optimizer)

    scheduler = get_linear_schedule_with_warmup(
        optimizer, num_warmup_steps=warmup_steps, num_training_steps=-1
    )

    # Crear DataLoader con la función de padding
    train_dataloader = DataLoader(
        dataset, 
        batch_size=batch_size, 
        shuffle=True, 
        collate_fn=collate_fn
    )

    for epoch in range(epochs):
        print(f"Training epoch {epoch}")

        for idx, input_tensors in tqdm(enumerate(train_dataloader)):
            input_tensors = input_tensors.to(device)
        
            # Forward pass
            outputs = model(input_tensors, labels=input_tensors)
        
            per_sample_loss = loss_function(outputs, input_tensors)  # 🔹 Usar función de pérdida corregida
        
            optimizer.zero_grad()
        
            optimizer.step(loss=per_sample_loss)  
        
            scheduler.step()

        if save_model_on_epoch:
            torch.save(
                model.state_dict(),
                os.path.join(output_dir, f"{output_prefix}-{epoch}.pt"),
            )

    return model


In [ ]:
dataset = CSVMIMIC("<|note|>", truncate=True, gpt2_type="gpt2")

In [ ]:
gpt2_type = "gpt2"

In [ ]:
YAML_boolq_string = """
group: multimedqa
task:
  - pubmedqa
aggregate_metric_list:
  - metric: acc
    aggregation: mean
    weight_by_size: True
"""
with open("multimedqa.yaml", "w") as f:
    f.write(YAML_boolq_string)

In [ ]:


model = GPT2LMHeadModel.from_pretrained("gpt2")
tokenizer = GPT2Tokenizer.from_pretrained(gpt2_type)
tokenizer.pad_token = tokenizer.eos_token
trained_model = train(dataset=dataset, model=model)

modelDir = 'model'
trained_model.save_pretrained(modelDir)
tokenizer=tokenizer
tokenizer.save_pretrained(modelDir)
result = !lm_eval \
--model hf \
--model_args pretrained=model \
--output_path /Output \
--include_path ./ \
--tasks multimedqa \
--device cpu \
--trust_remote_code



log_text = ' '.join(result)
match = re.search(r'pubmedqa\s+\|[\s\S]+?\|([\d.]+)', log_text[250900:])

if match:
    accuracy = float(match.group(1))
    print(f"Extracted Accuracy: {accuracy}")
else:
    accuracy = print("Accuracy not found in log.")


with open("results", "a") as f:
    f.write(f'{{ "accuracy": {accuracy} }}\n')



Training epoch 0


0it [00:01, ?it/s]


ValueError: Ghost clipping does not support parameter sharing. Parameter sharing may be due to default parameter sharing between lm_head and embedding.Please use a model without parameter sharing for ghost clipping.